In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import os
import json
import dotenv

from citation_extractor import extract_echr_citations
from llm import get_completion
from prompts import get_summarizer_prompt, get_analysis_prompt
from utils import load_json, save_json, normalize, find_closest_match


dotenv.load_dotenv()

True

In [3]:
data_path = '/Users/ahmed/Desktop/msc-24/ECHR_v2/echr_processed/'

df1_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_1.csv'
df2_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_2.csv'
df3_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_3.csv'
df4_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_4.csv'

df1 = pd.read_csv(df1_path)['file_path'].to_list()
df2 = pd.read_csv(df2_path)['file_path'].to_list()
df3 = pd.read_csv(df3_path)['file_path'].to_list()
df4 = pd.read_csv(df4_path)['file_path'].to_list()

id_to_name = load_json('/Users/ahmed/Desktop/msc-24/ECHR_v2/id_to_name.json')
name_to_id = {v: k for k, v in id_to_name.items()}
cases_names = list(id_to_name.values())

In [16]:
def do_that(df):
    results_ids = []
    results_paths = []
    results_names = []
    results_importance = [0,0,0,0]

    for i in range(125):
        case_path = data_path + df[i]
        case = load_json(case_path)
        text_list = case['law']

        #print(case['itemid'])

        results = extract_echr_citations(text_list)

        for item in results:
            cited_case_name = item['citation']['case_name']
            cited_case_name = normalize(cited_case_name)
            match_name = find_closest_match(cited_case_name, cases_names)
            cited_case_id = None
            try:
                cited_case_id = name_to_id[match_name]
            except:
                cited_case_id = None
            if cited_case_id:
                # add names
                results_names.append(match_name)
                # add id
                results_ids.append(cited_case_id)
                # add path
                cited_path = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed/' + cited_case_id + '.json'
                results_paths.append(cited_path)
                # add importance
                case = load_json(cited_path)
                results_importance[int(case['importance'])-1] = results_importance[int(case['importance'])-1] + 1
                
    return results_ids, results_paths, results_names, results_importance


In [17]:
results_ids_1, results_paths_1, results_names_1, results_importanc_1 = do_that(df1)
results_ids_2, results_paths_2, results_names_2, results_importanc_2 = do_that(df2)
results_ids_3, results_paths_3, results_names_3, results_importanc_3 = do_that(df3)
results_ids_4, results_paths_4, results_names_4, results_importanc_4 = do_that(df4)

In [18]:
results_importanc_1

[2159, 1108, 1345, 638]

In [19]:
results_importanc_2

[1245, 950, 908, 520]

In [20]:
results_importanc_3

[847, 356, 936, 443]

In [21]:
results_importanc_4

[115, 85, 432, 369]